# XAI Cấp độ 1 — CLIP + Spatial GAT trên Flickr30k Entities

Notebook này thực hiện toàn bộ các bước **B1 → B10** của kế hoạch XAI Cấp độ 1 và chạy trực tiếp trên dữ liệu,
mã nguồn và checkpoint mà nhóm đã bàn giao trong thư mục `0. final` trên Google Drive.

| Bước | Nội dung |
|---|---|
| B1 | Đóng băng đầu vào: checkpoint, α (đọc từ validation), split, checksum |
| B2 | Tái lập số test của 5 run, so với `final_test_summary.json` |
| B3 | Bảng thứ hạng theo từng truy vấn (CLIP, 3 seed, no_edge, random_edges) |
| B4 | Lõi XAI: tính điểm đồ thị với mặt nạ cạnh + lấy attention, có kiểm tra tính đúng |
| B5 | Bộ giải thích trung thực: đóng góp của từng quan hệ = mức giảm điểm khi bỏ quan hệ đó |
| B6 | Chọn số quan hệ giữ lại trong giải thích (K) trên **validation** |
| B7 | Gắn tên vùng từ Flickr30k Entities (loại câu truy vấn để tránh rò rỉ) |
| B8 | Chọn 5 ca thành công + 5 ca thất bại + 10 truy vấn ngẫu nhiên theo quy tắc cố định |
| B9 | Phân tích chi tiết 10 ca: hình + bảng số + nhận xét tự động |
| B10 | Kiểm chứng: bỏ quan hệ được giải thích so với bỏ quan hệ ngẫu nhiên, 3 seed |
| (Phụ) | Demo nhỏ: nhập truy vấn → top-k ảnh + giải thích |

**Nguyên tắc:** giải thích được tính từ chính hàm điểm của mô hình (mục 2 của đề). Không dùng test để chọn
bất kỳ tham số nào (mục 6). Mọi lựa chọn ca đều theo quy tắc viết sẵn, chạy lại cho ra đúng kết quả.

**Cách chạy:** Runtime → Change runtime type → GPU (T4 là đủ; CPU cũng chạy được nhưng chậm hơn). Sau đó Run all.
Nếu không tự tìm được thư mục `0. final`, hãy điền `FINAL_ROOT` ở ô cấu hình.

## 0. Cài đặt và cấu hình

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 2, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 3, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 4, globals())


## B1. Đóng băng đầu vào
Đọc α và epoch từ chính checkpoint (đã chọn trên validation). Ghi lại checksum để chứng minh dùng đúng file.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 6, globals())


## B2. Tái lập kết quả test
Dựng lại đồ thị bằng `graph_for` của nhóm, tính lại điểm, so với `final_test_summary.json`.
Nếu lệch quá ngưỡng thì dừng: mọi giải thích phía sau sẽ không còn đáng tin.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 8, globals())


## B3. Bảng thứ hạng theo từng truy vấn (test)
Thứ hạng = 1 + số ảnh có điểm lớn hơn ảnh đúng (cùng quy ước `ranking_metrics` của nhóm).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 10, globals())


## B4. Lõi XAI: điểm đồ thị có mặt nạ cạnh + attention

**Đơn vị giải thích là một quan hệ** = cặp cạnh hai chiều giữa hai nút (vì `graph_for` luôn thêm cả hai chiều).
Cạnh tự vòng luôn được giữ (là thông tin riêng của từng nút, không phải quan hệ).

Tên loại cạnh suy ra từ `relation_type` / `graph_for` (đã xác nhận): thông điệp đi từ nút nguồn → nút đích;
trục y của ảnh hướng xuống.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 12, globals())


## B5. Bộ giải thích trung thực (occlusion theo quan hệ)

Với truy vấn *q* và ảnh ứng viên *i*:
- `S_full` = điểm đồ thị gốc; `S_none` = điểm khi bỏ **mọi** quan hệ (chỉ còn tự vòng).
- **Đóng góp cấu trúc** = `S_full − S_none` (phần điểm đồ thị đến từ quan hệ, so với phần đến từ chính các nút).
- **Đóng góp của quan hệ g** = `Δ_g = S_full − S_full\g` (bỏ riêng quan hệ g, chạy lại 2 tầng GAT).
- **Giải thích** = tối đa K quan hệ có Δ dương lớn nhất (K chọn ở B6). Đường đi ≤ 2 bước về nút ảnh vì GAT có 2 tầng.
- Attention chỉ để hiển thị phụ: bộ mã hóa đồ thị **không nhìn thấy truy vấn**, nên attention giống nhau với mọi truy vấn.

Vì Δ được tính bằng cách chạy lại mô hình, mỗi quan hệ trong giải thích đều thực sự làm thay đổi điểm.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 14, globals())


## B6. Chọn K (số quan hệ trong giải thích) trên validation
Tiêu chí (comprehensiveness): gọi D(K) = mức giảm điểm đồ thị khi **xóa** K quan hệ có Δ dương lớn nhất,
D_all = mức giảm khi xóa **mọi** quan hệ có Δ dương. Chọn K nhỏ nhất sao cho trung vị D(K)/D_all ≥ 80%.
Không dùng tiêu chí "chỉ giữ K quan hệ" vì tỷ lệ đó có thể vượt 100% (bỏ luôn quan hệ đóng góp âm), không đơn điệu theo K.
Chỉ dùng validation, không chạm test.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 16, globals())


## B7. Gắn tên vùng từ Flickr30k Entities
Hộp vùng của nhóm chính là hộp gán tay của Flickr30k Entities (đã đối chiếu tọa độ). Mỗi vùng có `chain_ids`.
Tên vùng = loại (people, clothing, …) + một cụm từ lấy từ **các câu khác câu truy vấn**, để giải thích không
"mượn chữ" của chính truy vấn. Vùng chỉ được nhắc trong câu truy vấn thì chỉ hiện loại, kèm dấu `*`.
Tên vùng chỉ dùng để HIỂN THỊ, không tham gia tính điểm hay chọn quan hệ.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 18, globals())


## B7b. Ảnh gốc Flickr30k (chỉ để hiển thị)
Gói `0. final` chỉ có đặc trưng CLIP đã trích, không có ảnh gốc. Bước này tải bộ Flickr30k công khai trên Hugging Face
(`nlphuji/flickr30k`, khoảng 4 GB, chỉ tải một lần mỗi phiên Colab) và giải nén **đúng 1000 ảnh test** vào thư mục làm việc.
Ảnh chỉ dùng để vẽ hình và hiển thị trong demo, không tham gia tính điểm. Ảnh Flickr30k chỉ dùng cho nghiên cứu phi thương mại,
không phát tán lại ngoài nhóm. Nếu đã có ảnh, điền `IMAGE_DIR` ở ô cấu hình để bỏ qua bước tải.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 20, globals())


## B8. Chọn ca theo quy tắc cố định (trên bảng thứ hạng B3, mô hình chính seed 2026)

- **Thành công (5 ca):** CLIP xếp ảnh đúng > 1, CLIP + GAT xếp đúng = 1, **và** cả `no_edge` lẫn `random_edges`
  đều xếp > 1. Như vậy thành công có thể quy cho quan hệ thật, không phải cho phần adapter.
  Sắp theo mức tăng hạng giảm dần (hòa thì theo `caption_id`).
- **Thất bại (5 ca):** CLIP + GAT xếp ảnh đúng > 1. Lấy 3 ca "đồ thị làm hại" (CLIP = 1, GAT > 1; sụt hạng nhiều nhất)
  và 2 ca "không cứu được" (cả hai > 1; chọn ca suýt đúng: hạng GAT nhỏ nhất, rồi hạng CLIP lớn nhất).
  Thiếu loại nào thì bù bằng loại kia.
- Mỗi ảnh chỉ xuất hiện trong tối đa 1 ca (tránh 5 caption của cùng một ảnh chiếm chỗ).
- **Ngẫu nhiên (10 truy vấn):** rút bằng seed cố định, không nhìn kết quả — để tránh chỉ trưng ca đẹp.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 22, globals())


## B9. Phân tích chi tiết 10 ca (+ 10 truy vấn ngẫu nhiên)
Mỗi ca: giải thích cho **ảnh đúng**; với ca thất bại thêm giải thích cho **ảnh bị xếp nhầm lên đầu**;
với ca thành công thêm giải thích cho **ảnh CLIP từng xếp đầu** (để thấy vì sao nó bị đẩy xuống).
Hình: trái = bố cục vùng (vẽ lên ảnh thật nếu có `IMAGE_DIR`), phải = đồ thị con giải thích.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 24, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 25, globals())


## B10. Kiểm chứng: bỏ quan hệ được giải thích so với bỏ quan hệ ngẫu nhiên (3 seed)
Trên 20 truy vấn (5 thành công + 5 thất bại + 10 ngẫu nhiên), với từng checkpoint seed 2026/2027/2028:
- **Bỏ giải thích:** bỏ K quan hệ trong giải thích → mức giảm điểm đồ thị, điểm tổng và thay đổi hạng của ảnh đúng.
- **Bỏ ngẫu nhiên:** bỏ cùng số quan hệ nhưng chọn ngẫu nhiên, lặp 20 lần, lấy trung bình.
- **Chỉ giữ giải thích:** tỷ lệ đóng góp cấu trúc được giữ lại.
Báo cáo trung bình ± SD giữa 3 seed. Đây là bằng chứng giải thích đến từ cơ chế xếp hạng (mục 2 của đề).

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 27, globals())


## Tổng kết đầu ra
Tất cả nằm trong thư mục `OUT`:

| File | Nội dung |
|---|---|
| `B1_xai_config.json` | Checkpoint, checksum, α, K, phiên bản thư viện |
| `B2_reproduction_check.csv` | Tái lập số test so với nhóm |
| `B3_test_query_ranks.csv`, `B3_counts.csv` | Thứ hạng từng truy vấn và thống kê thành công/thất bại |
| `B4_core_checks.json` | Kiểm tra lõi XAI đúng cơ chế |
| `B6_val_k_selection.csv` | Chọn K trên validation |
| `B8_selected_cases.csv` | Danh sách ca theo quy tắc |
| `B9_cases.json`, `B9_case_analysis.md`, `cases/*.png` | Phân tích và hình từng ca |
| `B10_*.csv/json` | Kiểm chứng bỏ giải thích vs bỏ ngẫu nhiên, 3 seed |

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 29, globals())


## C1. Demo: nhập truy vấn bất kỳ → top-k kết quả + đồ thị giải thích tương tác

Đúng và chỉ đúng chức năng mục 8 của đề: *nhập truy vấn, trả về top-k kết quả và hiển thị quan hệ/đường đi giải thích*.
Giữ tiêu đề, mô tả, ô **Nhập câu truy vấn (Text Query)** và `launch(inline=False, share=True)` của notebook
`05_gat_xai_explanation_and_fidelity_v2` (ô 6); bố cục: **hàng truy vấn ở trên**, bên dưới là **2 tab rộng hết trang**.

- **Truy vấn bất kỳ** (tiếng Anh, vì CLIP huấn luyện trên tiếng Anh): mã hóa bằng đúng CLIP `ViT-B-32-quickgelu` /
  `openai` đóng băng, xếp hạng 1000 ảnh test bằng `final = (1 − α)·CLIP + α·Graph` (α từ checkpoint).
- **Tab Graph**: trình chiếu top-k ảnh gốc (nút ‹ ›, phím ← →, dải ảnh nhỏ). Trên ảnh, các vùng tham gia giải thích
  có khung tím kèm mã **R0, R1, …**; bên phải là **đồ thị giải thích tương tác kiểu Obsidian graph view của đúng ảnh
  đang xem** (đổi ảnh → đổi đồ thị). Nút tím là ảnh, nút xám là vùng (tên bắt đầu bằng cùng mã R), kích thước ∝ đóng
  góp Δ; cạnh liền + mũi tên là quan hệ trong giải thích, cạnh chấm là đường đi tiếp về ảnh. Rê chuột lên nút hoặc
  khung R: hai bên cùng sáng lên. Nút **⚙ Tùy chỉnh** (như Obsidian): bật/tắt nhãn quan hệ, tên nút, mũi tên, đường
  đi về ảnh; chỉnh kích thước nút, độ dày cạnh, lực đẩy, độ dài cạnh; căn vừa khung.
- **Tab Ranking**: bảng top-k (ảnh, final / CLIP / Graph, hạng nếu chỉ dùng CLIP để thấy GAT kéo ảnh lên/xuống); bấm một dòng để mở đồ thị của ảnh đó.
- Giải thích do **cùng hàm `explain()`** ở B5 sinh ra (tất định), nên cái hiển thị chính là cái đã kiểm chứng ở B10.
  Mỗi lần tìm kiếm được ghi vào `demo_explanations.jsonl`.

Các ca thành công/thất bại cho báo cáo là hình tĩnh ở B9.

In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 31, globals())


In [ ]:
from multimodal_xai.workflows.runner import run_stage
run_stage('06_flickr30k_gat_xai_level1', 32, globals())
